> 在线讲义：[第 20 章 · 计算单元：矩阵、向量、流水与低精度](https://wikieden.github.io/ai-infra-from-constraints/lessons/20-compute-units.html) · 本书《从约束推导 AI Infra》

# 第 20 章 · 计算单元：矩阵、向量、流水与低精度

配套讲义：仓库里 `ai-infra/notes/lessons/20-compute-units.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。纯 Python，不需要 GPU，也不下载模型。练习题不在这里算，先在讲义里自己做。

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 16³ 块 | $2 \times 16^3$ | `2 * 16**3` | 8192 |
| 第一步 MoE 补零 | 向上取整 | `padded_rows([2] * 256, 16)`、`padded_rows([64] * 8, 16)` | 4096、512 |
| 第二步 归约与 RoPE | $\log_2$、对数 × 6 | `reduce_rounds(128)`、`rope_flops(32, 8, 128)` | 7、15,360；表 256 KiB |
| 第三步 流水 | 三项取最大 | `attn_block_cycles(...)` | 1024 / 1024 / 768 / 512 |
| 第四步 量化存储 | $nb/8 + (n/g)s$ | `quant_mib(4096 * 4096, 4, 32, 1)` | 8.5 MiB；$g = 16$ 时 9 MiB |

16³ 矩阵块、MoE 补零、Softmax 归约、Qwen3-8B 的 RoPE、FlashAttention-4 的 B200 单 SM 分析、分组量化存储取自 AII 第 4 章 §4.2。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import math

**函数 `padded_rows`**：每个专家的行数向上补齐到 tile，返回实际执行的总行数。

In [ ]:
def padded_rows(rows_per_expert: list[int], tile: int = 16) -> int:
    """每个专家的行数向上补齐到 tile，返回实际执行的总行数。"""
    return sum(math.ceil(r / tile) * tile for r in rows_per_expert if r > 0)

**函数 `reduce_rounds`**：理想二叉归约要几轮。

In [ ]:
def reduce_rounds(n: int) -> int:
    """理想二叉归约要几轮。"""
    return math.ceil(math.log2(n))

**函数 `rope_flops`**：每 token 每层 RoPE：(n_q + n_k) × d_h/2 对，每对 4 乘 2 加。

In [ ]:
def rope_flops(n_q: int, n_k: int, d_h: int) -> int:
    """每 token 每层 RoPE：(n_q + n_k) × d_h/2 对，每对 4 乘 2 加。"""
    return (n_q + n_k) * (d_h // 2) * 6

**函数 `rope_table_kib`**：每个位置存 d_h/2 个 cos 和 d_h/2 个 sin。

In [ ]:
def rope_table_kib(positions: int, d_h: int = 128, b: int = 2) -> float:
    """每个位置存 d_h/2 个 cos 和 d_h/2 个 sin。"""
    return positions * d_h * b / 1024

**函数 `attn_block_cycles`**：FlashAttention-4 的 B200 单 SM 分析：(矩阵, 共享内存, 指数, 下界 τ) 周期。

In [ ]:
def attn_block_cycles(mat_x: float = 1, exp_x: float = 1, smem_x: float = 1, tile: int = 128, d: int = 128) -> tuple[float, float, float, float]:
    """FlashAttention-4 的 B200 单 SM 分析：(矩阵, 共享内存, 指数, 下界 τ) 周期。"""
    matrix = 2 * 2 * tile * tile * d / (8192 * mat_x)  # QK + PV 的 FLOPs ÷ 每周期矩阵速率
    exp = tile * tile / (16 * exp_x)  # 每个分数一次指数
    smem = 96 * 1024 / (128 * smem_x)  # 操作数从共享内存读 96 KiB
    return matrix, smem, exp, max(matrix, smem, exp)  # 流水下界取最慢的

**函数 `quant_mib`**：分组量化的存储量（MiB）= n·b/8 + (n/g)·s。

In [ ]:
def quant_mib(n: int, bits: int, group: int, scale_bytes: int) -> float:
    """分组量化的存储量（MiB）= n·b/8 + (n/g)·s。"""
    return (n * bits / 8 + n / group * scale_bytes) / 2**20

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步：矩阵单元")
print(f"  16×16×16 块：{2 * 16**3} FLOPs")
uni, con = padded_rows([2] * 256), padded_rows([64] * 8)
print(f"  MoE 512 个有效行：均匀 {uni} 行（有效 {512 / uni:.1%}），集中 {con} 行")

In [ ]:
print("\n第二步：向量工作")
print(f"  128 个数的归约 {reduce_rounds(128)} 轮")
print(f"  Qwen3-8B 一层 RoPE {rope_flops(32, 8, 128):,} FLOPs/token；1024 个位置的表 {rope_table_kib(1024):.0f} KiB")

In [ ]:
print("\n第三步：注意力流水（例 4-1）")
for name, kw in (("原样", {}), ("矩阵 ×2", {"mat_x": 2}), ("矩阵、指数 ×2", {"mat_x": 2, "exp_x": 2}), ("三项都 ×2", {"mat_x": 2, "exp_x": 2, "smem_x": 2})):
    m, s, e, tau = attn_block_cycles(**kw)
    print(f"  {name:10s} 矩阵 {m:.0f}，共享内存 {s:.0f}，指数 {e:.0f} → τ ≥ {tau:.0f}")

In [ ]:
print("\n第四步：4096×4096 权重的分组量化")
print(f"  4 bit、每 32 个一个 1 字节 scale：{quant_mib(4096 * 4096, 4, 32, 1):.1f} MiB；每 16 个：{quant_mib(4096 * 4096, 4, 16, 1):.1f} MiB；BF16 32 MiB")

## 3. 代码里才看得到的

1. `attn_block_cycles` 里 96 KiB 直接取自原文，没有从 Q、K、V 的形状推导；块形状改了，这个数也要重算。
2. `padded_rows` 只补行方向；真实 kernel 的列和 K 方向也可能补，补零比例会更低。
3. `quant_mib` 假设元素数能被分组大小整除；不整除时最后一组也要一个 scale。

## 4. 自己改一改

- 把 MoE 的 batch 从 64 改成 256，均匀分派时有效比例变成多少？
- 只把指数吞吐翻倍（矩阵不变），下界是多少？瓶颈在哪？
- 用 `quant_mib` 比较 FP8 不分组和 4 bit 每 128 个一组的存储量。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 零件传一排，领一次用多次；块太大行太少，补零也在算；
> 热身可并行，冠军要一轮轮比；
> 三个工位流水，出菜看最慢的；
> 缩印本轻，读前放大或戴放大镜，复印一次读多遍才划算。